# DataFrameIt - Exemplo 04: Prompt e coluna de texto

Este notebook mostra como o texto de cada linha chega ao prompt e como o DataFrameIt escolhe a coluna que contém esse texto.

**Conceitos demonstrados:**
- O marcador `{texto}` no prompt e o acréscimo automático quando ele falta
- Chaves que não são `{texto}` ficam no prompt como estão
- Inferência da coluna de texto e o parâmetro `text_column`
- Linhas sem texto, marcadas com "Texto ausente"

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bdcdo/dataframeit/blob/main/example/04_custom_placeholder.ipynb)

## 1. Instalação

In [ ]:
!pip install -q dataframeit[openai]

## 2. Configuração da API Key

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("API Key carregada dos Secrets do Colab")
except:
    pass

# os.environ['OPENAI_API_KEY'] = 'sua-chave-aqui'

if 'OPENAI_API_KEY' not in os.environ:
    print("AVISO: Configure sua OPENAI_API_KEY antes de continuar")
else:
    print("API Key configurada com sucesso!")

## 3. Imports

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
import pandas as pd
from dataframeit import dataframeit

## 4. Modelo Pydantic

O mesmo modelo simples serve para todos os exemplos deste notebook.

In [ ]:
class Avaliacao(BaseModel):
    """Classificação do sentimento de um comentário."""

    sentimento: Literal['positivo', 'negativo', 'neutro'] = Field(
        ...,
        description="Sentimento geral do comentário"
    )

## 5. Onde o texto entra no prompt

O DataFrameIt procura o marcador `{texto}` no prompt e o substitui pelo texto de cada linha. Assim você decide onde o texto aparece: no meio das instruções, entre delimitadores, antes das perguntas.

Se o prompt não tiver `{texto}`, o texto é acrescentado ao final, depois da linha "Texto a analisar:". Para prompts curtos, isso basta.

A célula abaixo só monta os dois prompts para você ver o que o modelo recebe. Ela não chama o modelo.

In [ ]:
PROMPT_COM_MARCADOR = """
Você é um analista de atendimento ao cliente.

Comentário do cliente:
<<<
{texto}
>>>

Classifique o sentimento do comentário acima.
"""

PROMPT_SEM_MARCADOR = "Classifique o sentimento do comentário do cliente."

df = pd.DataFrame({
    'id': [1, 2, 3],
    'texto': [
        "Entrega rápida e produto impecável.",
        "Veio quebrado e ninguém respondeu meus e-mails.",
        "Chegou no prazo, nada além disso.",
    ]
})

exemplo = df['texto'][0]

print("=== Com o marcador {texto} ===")
print(PROMPT_COM_MARCADOR.replace('{texto}', exemplo))

print("=== Sem o marcador: o texto vai para o final ===")
print(PROMPT_SEM_MARCADOR.rstrip() + "\n\nTexto a analisar:\n" + exemplo)

In [ ]:
resultado_com_marcador = dataframeit(df, Avaliacao, PROMPT_COM_MARCADOR)
resultado_sem_marcador = dataframeit(df, Avaliacao, PROMPT_SEM_MARCADOR)

print("Com o marcador:")
display(resultado_com_marcador[['id', 'sentimento']])

print("Sem o marcador:")
display(resultado_sem_marcador[['id', 'sentimento']])

## 6. Outras chaves ficam como estão

A substituição é literal: só a sequência `{texto}` é trocada. Qualquer outra chave no prompt chega ao modelo exatamente como foi escrita. Por isso você pode incluir um exemplo em JSON sem escapar as chaves.

A mesma regra tem um efeito colateral: um marcador com outro nome, como `{comentario}` ou `{review}`, **não** é preenchido. Ele vai ao modelo como texto literal, e o conteúdo da linha é acrescentado ao final, porque o prompt não tem `{texto}`. Use sempre `{texto}`.

In [ ]:
PROMPT_COM_JSON = """
Classifique o sentimento do comentário.

Exemplo de resposta esperada: {"sentimento": "positivo"}

Comentário:
{texto}
"""

print(PROMPT_COM_JSON.replace('{texto}', df['texto'][1]))

resultado_json = dataframeit(df, Avaliacao, PROMPT_COM_JSON)
resultado_json[['id', 'sentimento']]

## 7. Qual coluna vira texto

Num DataFrame, o parâmetro `text_column` diz qual coluna tem o texto. Com `text_column=None` (o padrão), o DataFrameIt tenta inferir:

1. Procura, nesta ordem, as colunas com os nomes da constante `TEXT_COLUMN_CANDIDATES` (impressa abaixo) e usa a primeira que existir. Se houver mais de uma, usa a primeira da ordem e emite um aviso.
2. Se nenhum desses nomes existir e o DataFrame tiver uma única coluna, usa essa coluna.
3. Caso contrário, levanta `ValueError` pedindo `text_column`.

Listas, dicionários e Series não precisam de `text_column`: o próprio valor é o texto (veja o notebook 07).

In [ ]:
from dataframeit.core import TEXT_COLUMN_CANDIDATES

print("Nomes procurados, em ordem:", TEXT_COLUMN_CANDIDATES)

# A coluna 'text' está entre os nomes procurados
df_text = pd.DataFrame({
    'id': [1, 2],
    'text': ["Adorei o atendimento.", "Demorou demais para chegar."],
})

# Uma única coluna, com qualquer nome
df_uma_coluna = pd.DataFrame({
    'mensagem': ["Produto excelente.", "Não funcionou na primeira semana."],
})

resultado_text = dataframeit(df_text, Avaliacao, PROMPT_COM_MARCADOR)
resultado_uma_coluna = dataframeit(df_uma_coluna, Avaliacao, PROMPT_COM_MARCADOR)

display(resultado_text[['id', 'text', 'sentimento']])
display(resultado_uma_coluna[['mensagem', 'sentimento']])

### Coluna com outro nome

Quando a coluna de texto tem um nome fora da lista e o DataFrame tem outras colunas, informe `text_column`.

In [ ]:
df_comentarios = pd.DataFrame({
    'id': [1, 2],
    'autor': ['Ana', 'Caio'],
    'comentario': ["Voltarei a comprar.", "Cobraram frete duas vezes."],
})

resultado_comentarios = dataframeit(
    df_comentarios,
    Avaliacao,
    PROMPT_COM_MARCADOR,
    text_column='comentario',
)

resultado_comentarios[['id', 'comentario', 'sentimento']]

### Quando não dá para inferir

Sem `text_column`, o mesmo DataFrame é ambíguo: nenhum nome da lista aparece e há mais de uma coluna. O erro sai antes de qualquer chamada ao modelo.

In [ ]:
try:
    dataframeit(df_comentarios, Avaliacao, PROMPT_COM_MARCADOR)
except ValueError as erro:
    print("ValueError:", erro)

## 8. Linhas sem texto

Uma linha cujo texto é vazio, `None` ou só espaços não vai ao modelo. Ela recebe status `'error'` e o detalhe "Texto ausente" em `_error_details`, e o DataFrameIt emite um aviso com a contagem.

As colunas `_dataframeit_status` e `_error_details` só aparecem no resultado quando alguma linha falhou ou registrou detalhe. Numa execução sem falhas elas são removidas, por isso o código confere se existem antes de exibi-las.

In [ ]:
df_com_vazios = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'texto': [
        "Ótimo custo-benefício.",
        "",
        None,
        "   ",
    ]
})

resultado_vazios = dataframeit(df_com_vazios, Avaliacao, PROMPT_COM_MARCADOR)

colunas = ['id', 'texto', 'sentimento']
colunas += [c for c in ('_dataframeit_status', '_error_details') if c in resultado_vazios.columns]
resultado_vazios[colunas]

## Resumo

- Escreva `{texto}` onde o texto deve aparecer; sem ele, o texto vai para o final do prompt
- Só `{texto}` é substituído, e as demais chaves chegam ao modelo como estão
- Em DataFrames, a coluna de texto é inferida por nome ou por ser a única; nos outros casos, use `text_column`
- Linhas sem texto não geram chamada e ficam marcadas com "Texto ausente"

---

## Próximos Passos

- [05_advanced_legal.ipynb](05_advanced_legal.ipynb) - Exemplo avançado com decisões judiciais
- [07_multiple_data_types.ipynb](07_multiple_data_types.ipynb) - Listas, dicionários e Series
- [02_error_handling.ipynb](02_error_handling.ipynb) - Tratamento de erros